In [1]:
import pandas as pd
import io
import base64
import re
import torch
from PIL import Image
from tqdm import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

def extract_option(model_output, true_answer):
    clean_output = model_output.strip().upper()
    if not clean_output: return "N/A"
    if clean_output in ["A", "B", "C", "D"]: return clean_output
    match = re.search(r"\b([A-D])\b", clean_output)
    if match: return match.group(1)
    true_ans_upper = str(true_answer).strip().upper()
    if true_ans_upper in clean_output:
        if not any(opt in clean_output for opt in ["A","B","C","D"] if opt != true_ans_upper):
            return true_ans_upper
    pure_chars = re.sub(r"[^\w]", "", clean_output)
    if pure_chars and pure_chars[0] in ["A","B","C","D"]: return pure_chars[0]
    return "N/A"

model_path = "./model_weights/qwen/Qwen3-VL-2B-Instruct"
input_file = "MMBench_DEV_EN.tsv"
output_file = "MMBench_5060_Final_Results.csv"
BATCH_SIZE = 4

min_pixels = 224 * 28 * 28
max_pixels = 448 * 28 * 28
processor = AutoProcessor.from_pretrained(model_path, min_pixels=min_pixels, max_pixels=max_pixels)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    attn_implementation="sdpa",
)
model.eval()

df = pd.read_csv(input_file, sep="\t")
all_results = []
correct_count = 0
processed_count = 0

def build_sample(row):
    image_obj = Image.open(io.BytesIO(base64.b64decode(row["image"]))).convert("RGB")
    options = f"\nA. {row['A']}\nB. {row['B']}"
    if pd.notna(row.get("C")): options += f"\nC. {row['C']}"
    if pd.notna(row.get("D")): options += f"\nD. {row['D']}"
    hint = f"Hint: {row['hint']}\n" if pd.notna(row.get("hint")) else ""
    prompt = f"{hint}Question: {row['question']}{options}\nAnswer with the option letter directly."
    messages = [{"role": "user", "content": [{"type": "image", "image": image_obj}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return image_obj, text

def run_single(row):
    img, text = build_sample(row)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, pad_token_id=processor.tokenizer.eos_token_id)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

rows = list(df.iterrows())
for batch_start in tqdm(range(0, len(rows), BATCH_SIZE)):
    batch_rows = [row for _, row in rows[batch_start:batch_start + BATCH_SIZE]]
    imgs, texts, valid_rows = [], [], []
    for row in batch_rows:
        try:
            img, text = build_sample(row)
            imgs.append(img)
            texts.append(text)
            valid_rows.append(row)
        except Exception:
            continue

    if not imgs:
        continue

    try:
        inputs = processor(text=texts, images=imgs, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=10, pad_token_id=processor.tokenizer.eos_token_id)
        in_len = inputs.input_ids.shape[1]
        outputs = [processor.decode(generated_ids[i][in_len:], skip_special_tokens=True) for i in range(len(valid_rows))]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        outputs = []
        for row in valid_rows:
            try:
                outputs.append(run_single(row))
            except Exception:
                outputs.append("")

    for row, model_output in zip(valid_rows, outputs):
        true_answer = str(row["answer"]).strip().upper()
        pred_char = extract_option(model_output, true_answer)
        is_correct = pred_char == true_answer
        if is_correct: correct_count += 1
        processed_count += 1
        all_results.append({
            "index": row["index"],
            "true_answer": true_answer,
            "pred_answer": pred_char,
            "is_correct": is_correct,
            "model_output_raw": model_output,
        })

accuracy = correct_count / processed_count * 100 if processed_count else 0
print(f"Accuracy: {accuracy:.2f}% ({correct_count}/{processed_count})")
pd.DataFrame(all_results).to_csv(output_file, index=False, encoding="utf-8-sig")


c:\Users\Lenovo\.conda\envs\qwen3_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1083/1083 [02:24<00:00,  7.52it/s]

Accuracy: 37.03% (431/1164)
